# 06 — Household Amenities & Development Index
**Source:** India Districts Census 2011 · 640 districts  
Covers electricity, LPG, sanitation, internet, and a composite Development Index.

In [ ]:
import sqlite3, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi'] = 130

DB = '../data/processed/census_model.db'
conn = sqlite3.connect(DB)

df = pd.read_sql("""
    SELECT d.state_name, d.district_name,
           f.population, f.households,
           f.housholds_with_electric_lighting,
           f.lpg_or_png_households,
           f.households_with_internet,
           f.households_with_computer,
           f.having_latrine_facility_within_the_premises_total_households,
           f.having_bathing_facility_total_households,
           f.households_with_separate_kitchen_cooking_inside_house,
           f.not_having_latrine_facility_within_the_premises_alternative_source_open_households,
           f.literate, f.literate_education, f.total_education,
           f.power_parity_above_rs_545000,
           f.power_parity_rs_330000_545000,
           f.total_power_parity
    FROM fact_census f
    JOIN dim_location d ON f.location_id = d.location_id
""", conn)
conn.close()

df = df[df['households'] > 0]

df['electricity_pct'] = (df['housholds_with_electric_lighting'] / df['households'] * 100).round(2)
df['lpg_pct']         = (df['lpg_or_png_households'] / df['households'] * 100).round(2)
df['internet_pct']    = (df['households_with_internet'] / df['households'] * 100).round(2)
df['latrine_pct']     = (df['having_latrine_facility_within_the_premises_total_households'] / df['households'] * 100).round(2)
df['bathing_pct']     = (df['having_bathing_facility_total_households'] / df['households'] * 100).round(2)
df['kitchen_pct']     = (df['households_with_separate_kitchen_cooking_inside_house'] / df['households'] * 100).round(2)
df['open_defecation_pct'] = (df['not_having_latrine_facility_within_the_premises_alternative_source_open_households'] / df['households'] * 100).round(2)
df['literacy_rate']   = (df['literate'] / df['population'] * 100).round(2)

high_income_pct = pd.Series(np.where(
    df['total_power_parity'] > 0,
    df['power_parity_above_rs_545000'] / df['total_power_parity'] * 100,
    0
), index=df.index)

# Composite Development Index: equal-weight average of 5 normalized metrics
def norm(s): return (s - s.min()) / (s.max() - s.min()) * 100

df['dev_index'] = (
    norm(df['electricity_pct']) +
    norm(df['lpg_pct']) +
    norm(df['latrine_pct']) +
    norm(df['literacy_rate']) +
    norm(high_income_pct)
) / 5
df['dev_index'] = df['dev_index'].round(2)

state = df.groupby('state_name').agg(
    households=('households','sum'),
    electricity=('housholds_with_electric_lighting','sum'),
    lpg=('lpg_or_png_households','sum'),
    internet=('households_with_internet','sum'),
    latrine=('having_latrine_facility_within_the_premises_total_households','sum'),
    bathing=('having_bathing_facility_total_households','sum'),
    kitchen=('households_with_separate_kitchen_cooking_inside_house','sum'),
    open_def=('not_having_latrine_facility_within_the_premises_alternative_source_open_households','sum'),
).reset_index()

for col, name in [('electricity','electricity_pct'),('lpg','lpg_pct'),('internet','internet_pct'),
                   ('latrine','latrine_pct'),('bathing','bathing_pct'),('kitchen','kitchen_pct'),
                   ('open_def','open_defecation_pct')]:
    state[name] = (state[col] / state['households'] * 100).round(2)

print('Setup complete. Districts:', len(df))

## 6.1 — Composite Development Index — Top & Bottom 15 Districts

In [ ]:
top15    = df.nlargest(15, 'dev_index')[['district_name','state_name','dev_index']]
bottom15 = df.nsmallest(15, 'dev_index')[['district_name','state_name','dev_index']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

ax1.barh(top15['district_name'] + ', '+ top15['state_name'], top15['dev_index'],
         color=sns.color_palette('Greens_r', 15))
ax1.set_xlabel('Development Index (0–100)')
ax1.set_title('Top 15 Most Developed Districts', fontweight='bold')
ax1.invert_yaxis()

ax2.barh(bottom15['district_name'] + ', '+ bottom15['state_name'], bottom15['dev_index'],
         color=sns.color_palette('Reds_r', 15))
ax2.set_xlabel('Development Index (0–100)')
ax2.set_title('Bottom 15 Least Developed Districts', fontweight='bold')
ax2.invert_yaxis()

plt.suptitle('Composite Development Index\n(Electricity + LPG + Latrine + Literacy + High Income — normalized average)',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

## 6.2 — Amenity Access Heatmap by State

In [ ]:
amenity_cols  = ['electricity_pct','lpg_pct','latrine_pct','bathing_pct','kitchen_pct','internet_pct']
amenity_labels= ['Electricity','LPG/PNG','Latrine (in-premises)','Bathing Facility','Separate Kitchen','Internet']

heat = state.set_index('state_name')[amenity_cols].copy()
heat.columns = amenity_labels
heat = heat.sort_values('Electricity', ascending=False)

fig, ax = plt.subplots(figsize=(12, 12))
sns.heatmap(heat, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=.4,
            cbar_kws={'label': '% of Households'}, ax=ax)
ax.set_title('Household Amenity Access by State (%)\nsorted by Electricity Access',
             fontsize=12, fontweight='bold')
ax.set_xlabel('')
plt.tight_layout(); plt.show()

## 6.3 — Open Defecation Rate by State (Sanitation Deficit)

In [ ]:
state_od = state.sort_values('open_defecation_pct', ascending=False)

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#e74c3c' if v > 50 else '#f39c12' if v > 25 else '#3498db' for v in state_od['open_defecation_pct']]
bars = ax.barh(state_od['state_name'], state_od['open_defecation_pct'], color=colors)
ax.set_xlabel('% of Households with Open Defecation')
ax.set_title('Open Defecation Rate by State\n(Red >50% | Orange 25–50% | Blue ≤25%)',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars, state_od['open_defecation_pct']):
    ax.text(bar.get_width() + 0.4, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=8)
plt.tight_layout(); plt.show()

## 6.4 — Electricity vs LPG Access (District Scatter)

In [ ]:
fig = px.scatter(
    df, x='electricity_pct', y='lpg_pct',
    color='state_name', hover_name='district_name',
    hover_data={'state_name': True, 'electricity_pct': ':.1f', 'lpg_pct': ':.1f'},
    labels={'electricity_pct': 'Electricity Access (%)', 'lpg_pct': 'LPG/PNG Access (%)'},
    title='Electricity Access vs LPG/PNG Access — District Level',
    trendline='ols', trendline_scope='overall', trendline_color_override='black'
)
fig.update_layout(showlegend=False, height=550)
fig.show()

## 6.5 — Development Index Distribution (Histogram + KDE)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df['dev_index'], bins=40, kde=True, color='#3498db', ax=ax)
ax.axvline(df['dev_index'].mean(), color='red', linestyle='--',
           label=f"Mean: {df['dev_index'].mean():.1f}")
ax.axvline(df['dev_index'].median(), color='green', linestyle='--',
           label=f"Median: {df['dev_index'].median():.1f}")
ax.set_xlabel('Development Index (0–100)')
ax.set_ylabel('Number of Districts')
ax.set_title('Distribution of Composite Development Index Across 640 Districts',
             fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()